# TokenMM Live Trading Report

Load local SQLite telemetry for live TokenMM fills, order actions, quote cycles, and markouts. The report now leads with a single presentation table of average resolved markouts in basis points, plus a short interpretation block.


In [1]:
from pathlib import Path
import os
import sqlite3

import pandas as pd

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 120
pd.options.display.width = 220

TELEMETRY_DIR = Path(
    os.environ.get("TOKENMM_TELEMETRY_DIR", "/var/lib/nautilus/telemetry/tokenmm")
)
DB_PATHS = {
    "execution_fill": TELEMETRY_DIR / "fills.sqlite",
    "order_action": TELEMETRY_DIR / "orders.sqlite",
    "quote_cycle": TELEMETRY_DIR / "quote_cycles.sqlite",
    "execution_markout": TELEMETRY_DIR / "markouts.sqlite",
}
DB_PATHS


{'execution_fill': PosixPath('/var/lib/nautilus/telemetry/tokenmm/fills.sqlite'),
 'order_action': PosixPath('/var/lib/nautilus/telemetry/tokenmm/orders.sqlite'),
 'quote_cycle': PosixPath('/var/lib/nautilus/telemetry/tokenmm/quote_cycles.sqlite'),
 'execution_markout': PosixPath('/var/lib/nautilus/telemetry/tokenmm/markouts.sqlite')}

In [2]:
def read_sqlite_table(path: Path, table: str, limit: int | None = None) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    query = f"SELECT * FROM {table}"
    if limit is not None:
        query += f" ORDER BY rowid DESC LIMIT {int(limit)}"
    with sqlite3.connect(path) as conn:
        return pd.read_sql_query(query, conn)


def read_sqlite_query(path: Path, query: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    with sqlite3.connect(path) as conn:
        return pd.read_sql_query(query, conn)


def ns_to_utc(series: pd.Series) -> pd.Series:
    return pd.to_datetime(pd.to_numeric(series, errors="coerce"), unit="ns", utc=True)


def ms_to_utc(series: pd.Series) -> pd.Series:
    return pd.to_datetime(pd.to_numeric(series, errors="coerce"), unit="ms", utc=True)


def numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def format_utc(series: pd.Series) -> pd.Series:
    return series.dt.strftime("%Y-%m-%d %H:%M:%S UTC").fillna("n/a")


def format_number(series: pd.Series, digits: int = 6, suffix: str = "") -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    return values.map(lambda value: f"{value:.{digits}f}{suffix}" if pd.notna(value) else "n/a")


def format_signed_number(series: pd.Series, digits: int = 2, suffix: str = "") -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    return values.map(lambda value: f"{value:+.{digits}f}{suffix}" if pd.notna(value) else "n/a")


def summarize_sources(series: pd.Series) -> str:
    unique_values = sorted({str(value) for value in series.dropna() if str(value).strip()})
    return ", ".join(unique_values) if unique_values else "n/a"


In [3]:
fills = read_sqlite_table(DB_PATHS["execution_fill"], "execution_fill")
markouts = read_sqlite_table(DB_PATHS["execution_markout"], "execution_markout")
orders_recent = read_sqlite_query(
    DB_PATHS["order_action"],
    """
    SELECT strategy_id, instrument_id, action_type, action_state, client_order_id, venue_order_id,
           quote_cycle_id, reason_code, ts_event
    FROM order_action
    ORDER BY ts_event DESC
    LIMIT 50000
    """,
)
order_action_overview = read_sqlite_query(
    DB_PATHS["order_action"],
    "SELECT COUNT(*) AS rows, MAX(ts_event) AS latest_ts_event_ns FROM order_action",
)
quote_cycle_overview = read_sqlite_query(
    DB_PATHS["quote_cycle"],
    "SELECT COUNT(*) AS rows, MAX(ts_cycle_start_ns) AS latest_ts_cycle_start_ns FROM quote_cycle",
)

fills_enriched = fills.assign(
    fill_ts_utc=ns_to_utc(fills["ts_event"]),
    last_qty_num=numeric(fills["last_qty"]),
    last_px_num=numeric(fills["last_px"]),
)
orders_enriched = orders_recent.assign(ts_event_utc=ns_to_utc(orders_recent["ts_event"]))
markouts_enriched = markouts.assign(
    target_ts_utc=ms_to_utc(markouts["target_ts_ms"]),
    benchmark_ts_utc=ms_to_utc(markouts["benchmark_ts_ms"]),
    fill_qty_num=numeric(markouts["fill_qty"]),
    fill_px_num=numeric(markouts["fill_px"]),
    benchmark_px_num=numeric(markouts["benchmark_px"]),
    markout_abs_num=numeric(markouts["markout_abs"]),
    markout_bps_num=numeric(markouts["markout_bps"]),
    fv_source=markouts["benchmark_name"].fillna("n/a"),
)

telemetry_overview = pd.DataFrame(
    [
        {
            "table_name": "execution_fill",
            "rows": len(fills_enriched),
            "latest_utc": fills_enriched["fill_ts_utc"].max(),
        },
        {
            "table_name": "order_action",
            "rows": int(order_action_overview.loc[0, "rows"]),
            "latest_utc": ns_to_utc(order_action_overview["latest_ts_event_ns"]).iloc[0],
        },
        {
            "table_name": "quote_cycle",
            "rows": int(quote_cycle_overview.loc[0, "rows"]),
            "latest_utc": ns_to_utc(quote_cycle_overview["latest_ts_cycle_start_ns"]).iloc[0],
        },
        {
            "table_name": "execution_markout",
            "rows": len(markouts_enriched),
            "latest_utc": markouts_enriched["target_ts_utc"].max(),
        },
    ]
)
telemetry_overview["latest_utc"] = format_utc(pd.to_datetime(telemetry_overview["latest_utc"], utc=True, errors="coerce"))
telemetry_overview


,table_name,rows,latest_utc
0,execution_fill,608,2026-03-12 08:09:17 UTC
1,order_action,3493591,2026-03-12 08:46:28 UTC
2,quote_cycle,1643134,2026-03-12 08:46:33 UTC
3,execution_markout,306,2026-03-12 08:11:17 UTC


## Strategy Snapshot

Use this for the top-level readout. It shows fills, markout coverage, FV source, and average resolved markout performance by strategy without raw null-heavy rows.


In [4]:
strategy_fills = (
    fills_enriched.groupby("strategy_id", dropna=False)
    .agg(
        fills=("trade_id", "nunique"),
        last_fill_utc=("fill_ts_utc", "max"),
    )
)

strategy_markouts = (
    markouts_enriched.groupby("strategy_id", dropna=False)
    .agg(
        fv_source=("fv_source", summarize_sources),
        markout_rows=("trade_id", "count"),
        resolved_rows=("resolution_status", lambda s: int((s == "resolved").sum())),
        expired_rows=("resolution_status", lambda s: int((s == "expired").sum())),
        avg_resolved_markout_bps=("markout_bps_num", "mean"),
        last_markout_target_utc=("target_ts_utc", "max"),
    )
)

strategy_snapshot = strategy_fills.join(strategy_markouts, how="outer").reset_index()
strategy_snapshot["fills"] = strategy_snapshot["fills"].fillna(0).astype(int)
strategy_snapshot["markout_rows"] = strategy_snapshot["markout_rows"].fillna(0).astype(int)
strategy_snapshot["resolved_rows"] = strategy_snapshot["resolved_rows"].fillna(0).astype(int)
strategy_snapshot["expired_rows"] = strategy_snapshot["expired_rows"].fillna(0).astype(int)
strategy_snapshot["fv_source"] = strategy_snapshot["fv_source"].fillna("n/a")
strategy_snapshot["resolution_rate_pct"] = (
    strategy_snapshot["resolved_rows"]
    .div(strategy_snapshot["markout_rows"].replace(0, pd.NA))
    .mul(100)
)

strategy_snapshot_display = strategy_snapshot.assign(
    last_fill_utc=format_utc(pd.to_datetime(strategy_snapshot["last_fill_utc"], utc=True, errors="coerce")),
    last_markout_target_utc=format_utc(pd.to_datetime(strategy_snapshot["last_markout_target_utc"], utc=True, errors="coerce")),
    resolution_rate_pct=format_number(strategy_snapshot["resolution_rate_pct"], digits=1, suffix="%"),
    avg_resolved_markout_bps=format_signed_number(strategy_snapshot["avg_resolved_markout_bps"], digits=2, suffix=" bps"),
)[[
    "strategy_id",
    "fv_source",
    "fills",
    "markout_rows",
    "resolved_rows",
    "expired_rows",
    "resolution_rate_pct",
    "avg_resolved_markout_bps",
    "last_fill_utc",
    "last_markout_target_utc",
]].sort_values(["fills", "markout_rows", "strategy_id"], ascending=[False, False, True])

strategy_snapshot_display


,strategy_id,fv_source,fills,markout_rows,resolved_rows,expired_rows,resolution_rate_pct,avg_resolved_markout_bps,last_fill_utc,last_markout_target_utc
1,plumeusdt_bybit_perp_makerv3-000,fv_market_mid,220,108,42,66,38.9%,+20.40 bps,2026-03-12 08:09:17 UTC,2026-03-12 08:11:17 UTC
2,plumeusdt_bybit_spot_makerv3-000,fv_market_mid,200,59,15,44,25.4%,+19.32 bps,2026-03-12 08:09:17 UTC,2026-03-12 08:11:17 UTC
3,plumeusdt_okx_perp_makerv3-000,fv_market_mid,175,100,30,70,30.0%,-3.79 bps,2026-03-12 06:47:45 UTC,2026-03-12 06:49:45 UTC
0,plumeusdt_bitget_perp_makerv3-000,fv_market_mid,13,39,0,39,0.0%,n/a,2026-03-12 00:20:29 UTC,2026-03-12 00:22:29 UTC


## Markout Summary to Present

This is the one-table summary for presentation. Each row is a strategy, and each column is a `horizon | source` combination. The values are average resolved markouts in basis points.


In [5]:
markout_summary_long = (
    markouts_enriched.groupby(["strategy_id", "horizon_s", "fv_source"], dropna=False)
    .agg(
        avg_markout_bps=("markout_bps_num", "mean"),
        resolved_rows=("resolution_status", lambda s: int((s == "resolved").sum())),
    )
    .reset_index()
)
markout_summary_long["summary_column"] = (
    markout_summary_long["horizon_s"].astype(int).astype(str)
    + "s | "
    + markout_summary_long["fv_source"].astype(str)
)

column_order = (
    markout_summary_long[["horizon_s", "summary_column"]]
    .drop_duplicates()
    .sort_values(["horizon_s", "summary_column"])["summary_column"]
    .tolist()
)

markout_summary_display = (
    markout_summary_long.pivot(index="strategy_id", columns="summary_column", values="avg_markout_bps")
    .reindex(columns=column_order)
    .sort_index()
)
markout_summary_display = markout_summary_display.apply(
    lambda column: format_signed_number(column, digits=2, suffix=" bps")
)
markout_summary_display.index.name = "strategy_id"
markout_summary_display


summary_column,30s | fv_market_mid,60s | fv_market_mid,120s | fv_market_mid
strategy_id,,,
plumeusdt_bitget_perp_makerv3-000,n/a,n/a,n/a
plumeusdt_bybit_perp_makerv3-000,+19.65 bps,+21.33 bps,+20.22 bps
plumeusdt_bybit_spot_makerv3-000,+19.33 bps,+17.26 bps,+21.38 bps
plumeusdt_okx_perp_makerv3-000,+0.89 bps,-4.80 bps,-7.45 bps


In [6]:
overall_source = summarize_sources(markouts_enriched["fv_source"])
portfolio_horizon_bps = (
    markouts_enriched.groupby("horizon_s", dropna=False)["markout_bps_num"]
    .mean()
    .sort_index()
)

resolved_rank = markout_summary_long.loc[markout_summary_long["avg_markout_bps"].notna()].sort_values("avg_markout_bps", ascending=False)
summary_lines = [f"FV source in this sample: {overall_source}."]
summary_lines.append(
    "Average resolved markout by horizon across all strategies: "
    + "; ".join(
        f"{int(horizon)}s {value:+.2f} bps"
        for horizon, value in portfolio_horizon_bps.items()
        if pd.notna(value)
    )
    + "."
)
if not resolved_rank.empty:
    best = resolved_rank.iloc[0]
    worst = resolved_rank.iloc[-1]
    summary_lines.append(
        f"Best resolved reading in the sample: {best['strategy_id']} at {int(best['horizon_s'])}s ({best['avg_markout_bps']:+.2f} bps)."
    )
    summary_lines.append(
        f"Weakest resolved reading in the sample: {worst['strategy_id']} at {int(worst['horizon_s'])}s ({worst['avg_markout_bps']:+.2f} bps)."
    )

unresolved_strategies = sorted(
    markout_summary_long.loc[markout_summary_long["resolved_rows"] == 0, "strategy_id"].unique().tolist()
)
if unresolved_strategies:
    summary_lines.append(
        "No resolved markouts yet for: " + ", ".join(unresolved_strategies) + "."
    )

markout_summary_text = chr(10).join(summary_lines)
print(markout_summary_text)


FV source in this sample: fv_market_mid.
Average resolved markout by horizon across all strategies: 30s +13.13 bps; 60s +11.62 bps; 120s +10.88 bps.
Best resolved reading in the sample: plumeusdt_bybit_spot_makerv3-000 at 120s (+21.38 bps).
Weakest resolved reading in the sample: plumeusdt_okx_perp_makerv3-000 at 120s (-7.45 bps).
No resolved markouts yet for: plumeusdt_bitget_perp_makerv3-000.


## Appendix: Recent Resolved Markouts

Concrete examples of the newest trades where a post-fill FV benchmark arrived and the markout resolved successfully.


In [7]:
recent_resolved_markouts = (
    markouts_enriched.loc[markouts_enriched["resolution_status"] == "resolved"]
    .sort_values(["target_ts_utc", "trade_id", "horizon_s"], ascending=[False, True, True])
    .merge(
        fills_enriched[["trade_id", "fill_ts_utc", "liquidity_side"]].drop_duplicates(),
        on="trade_id",
        how="left",
    )
)
recent_resolved_markouts_display = recent_resolved_markouts.assign(
    fill_ts_utc=format_utc(pd.to_datetime(recent_resolved_markouts["fill_ts_utc"], utc=True, errors="coerce")),
    target_ts_utc=format_utc(pd.to_datetime(recent_resolved_markouts["target_ts_utc"], utc=True, errors="coerce")),
    benchmark_ts_utc=format_utc(pd.to_datetime(recent_resolved_markouts["benchmark_ts_utc"], utc=True, errors="coerce")),
    fill_qty=format_number(recent_resolved_markouts["fill_qty_num"], digits=2),
    fill_px=format_number(recent_resolved_markouts["fill_px_num"], digits=8),
    benchmark_px=format_number(recent_resolved_markouts["benchmark_px_num"], digits=8),
    markout_abs=format_number(recent_resolved_markouts["markout_abs_num"], digits=8),
    markout_bps=format_signed_number(recent_resolved_markouts["markout_bps_num"], digits=2, suffix=" bps"),
)[[
    "strategy_id",
    "instrument_id",
    "fv_source",
    "trade_id",
    "client_order_id",
    "horizon_s",
    "liquidity_side",
    "fill_qty",
    "fill_px",
    "benchmark_px",
    "markout_abs",
    "markout_bps",
    "fill_ts_utc",
    "benchmark_ts_utc",
    "target_ts_utc",
]].head(20)

recent_resolved_markouts_display


,strategy_id,instrument_id,fv_source,trade_id,client_order_id,horizon_s,liquidity_side,fill_qty,fill_px,benchmark_px,markout_abs,markout_bps,fill_ts_utc,benchmark_ts_utc,target_ts_utc
0,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,fv_market_mid,2270000001366793465,O-20260312-080909-SPOT-000-158,120,MAKER,1000.00,0.01211000,0.01213000,0.00002000,+16.52 bps,2026-03-12 08:09:17 UTC,2026-03-12 08:11:17 UTC,2026-03-12 08:11:17 UTC
1,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,fv_market_mid,2270000001366793464,O-20260312-080909-SPOT-000-157,120,MAKER,1000.00,0.01212000,0.01213000,0.00001000,+8.25 bps,2026-03-12 08:09:17 UTC,2026-03-12 08:11:17 UTC,2026-03-12 08:11:17 UTC
2,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,5d68cd58-ccf4-5c72-9369-a1bef89f4e53,O-20260312-080910-PERP-000-4807,120,MAKER,1000.00,0.01210000,0.01212225,0.00002225,+18.39 bps,2026-03-12 08:09:17 UTC,2026-03-12 08:11:17 UTC,2026-03-12 08:11:17 UTC
3,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,889746cc-153b-5ab1-86fa-255788948a24,O-20260312-080910-PERP-000-4806,120,MAKER,1000.00,0.01210600,0.01212225,0.00001625,+13.42 bps,2026-03-12 08:09:17 UTC,2026-03-12 08:11:17 UTC,2026-03-12 08:11:17 UTC
4,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,fv_market_mid,2270000001366793458,O-20260312-080910-SPOT-000-163,120,MAKER,1000.00,0.01213000,0.01213000,0.00000000,+0.00 bps,2026-03-12 08:09:17 UTC,2026-03-12 08:11:17 UTC,2026-03-12 08:11:17 UTC
5,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,31d943bb-7727-544f-b034-974caadbd34d,O-20260312-080910-PERP-000-4805,120,MAKER,1000.00,0.01211200,0.01212225,0.00001025,+8.46 bps,2026-03-12 08:09:17 UTC,2026-03-12 08:11:17 UTC,2026-03-12 08:11:17 UTC
6,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,9d834c64-ef28-52c7-8451-a80813a371db,O-20260312-080910-PERP-000-4804,120,MAKER,1000.00,0.01211800,0.01212225,0.00000425,+3.51 bps,2026-03-12 08:09:17 UTC,2026-03-12 08:11:17 UTC,2026-03-12 08:11:17 UTC
7,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,f4040019-c8cb-53b9-8898-7449f7387cfd,O-20260312-080854-PERP-000-4654,120,MAKER,1000.00,0.01212900,0.01212375,-0.00000525,-4.33 bps,2026-03-12 08:09:02 UTC,2026-03-12 08:11:02 UTC,2026-03-12 08:11:02 UTC
8,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,fv_market_mid,2270000001366793258,O-20260312-080851-SPOT-000-132,120,MAKER,1000.00,0.01218000,0.01212500,0.00005500,+45.16 bps,2026-03-12 08:08:54 UTC,2026-03-12 08:10:55 UTC,2026-03-12 08:10:54 UTC
9,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,2a986ff8-70af-5539-ad6c-fb793de14edd,O-20260312-080846-PERP-000-4588,120,MAKER,1000.00,0.01217400,0.01211825,0.00005575,+45.79 bps,2026-03-12 08:08:54 UTC,2026-03-12 08:10:54 UTC,2026-03-12 08:10:54 UTC


## Appendix: Recent Expired Markouts

Operational follow-up view for rows that were persisted but did not receive a usable benchmark before expiry.


In [8]:
recent_expired_markouts = (
    markouts_enriched.loc[markouts_enriched["resolution_status"] == "expired"]
    .sort_values(["target_ts_utc", "trade_id", "horizon_s"], ascending=[False, True, True])
    .merge(
        fills_enriched[["trade_id", "fill_ts_utc", "liquidity_side"]].drop_duplicates(),
        on="trade_id",
        how="left",
    )
)
recent_expired_markouts_display = recent_expired_markouts.assign(
    fill_ts_utc=format_utc(pd.to_datetime(recent_expired_markouts["fill_ts_utc"], utc=True, errors="coerce")),
    target_ts_utc=format_utc(pd.to_datetime(recent_expired_markouts["target_ts_utc"], utc=True, errors="coerce")),
)[[
    "strategy_id",
    "instrument_id",
    "fv_source",
    "trade_id",
    "client_order_id",
    "horizon_s",
    "liquidity_side",
    "fill_ts_utc",
    "target_ts_utc",
]].head(20)

recent_expired_markouts_display


,strategy_id,instrument_id,fv_source,trade_id,client_order_id,horizon_s,liquidity_side,fill_ts_utc,target_ts_utc
0,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,fv_market_mid,2270000001366728756,O-20260312-053830-SPOT-000-3908,60,MAKER,2026-03-12 05:38:33 UTC,2026-03-12 05:39:33 UTC
1,plumeusdt_okx_perp_makerv3-000,PLUME-USDT-SWAP.OKX,fv_market_mid,26119854,O20260312053845PERP00011847,30,MAKER,2026-03-12 05:38:51 UTC,2026-03-12 05:39:21 UTC
2,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,fv_market_mid,2270000001366728756,O-20260312-053830-SPOT-000-3908,30,MAKER,2026-03-12 05:38:33 UTC,2026-03-12 05:39:03 UTC
3,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,9eef597e-e239-5123-8412-307364ed2d48,O-20260312-050239-PERP-000-32621,120,MAKER,2026-03-12 05:02:48 UTC,2026-03-12 05:04:48 UTC
4,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,e6e7fc15-dade-50d0-b9b2-e265e1166419,O-20260312-050226-PERP-000-32511,120,MAKER,2026-03-12 05:02:29 UTC,2026-03-12 05:04:29 UTC
5,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,9eef597e-e239-5123-8412-307364ed2d48,O-20260312-050239-PERP-000-32621,60,MAKER,2026-03-12 05:02:48 UTC,2026-03-12 05:03:48 UTC
6,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,e6e7fc15-dade-50d0-b9b2-e265e1166419,O-20260312-050226-PERP-000-32511,60,MAKER,2026-03-12 05:02:29 UTC,2026-03-12 05:03:29 UTC
7,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,9eef597e-e239-5123-8412-307364ed2d48,O-20260312-050239-PERP-000-32621,30,MAKER,2026-03-12 05:02:48 UTC,2026-03-12 05:03:18 UTC
8,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,e6e7fc15-dade-50d0-b9b2-e265e1166419,O-20260312-050226-PERP-000-32511,30,MAKER,2026-03-12 05:02:29 UTC,2026-03-12 05:02:59 UTC
9,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,fv_market_mid,0a0e1136-ec7e-59c5-a975-4d59884f70cc,O-20260312-045345-PERP-000-30241,120,MAKER,2026-03-12 04:53:45 UTC,2026-03-12 04:55:45 UTC


## Appendix: Recent Fills With Markout Status

Recent execution rows joined to order context and current markout coverage for one-table troubleshooting.


In [9]:
markout_status_by_trade = (
    markouts_enriched.groupby("trade_id", dropna=False)
    .agg(
        fv_source=("fv_source", summarize_sources),
        markout_rows=("trade_id", "count"),
        resolved_rows=("resolution_status", lambda s: int((s == "resolved").sum())),
        expired_rows=("resolution_status", lambda s: int((s == "expired").sum())),
        best_resolved_markout_bps=("markout_bps_num", "mean"),
    )
    .reset_index()
)

recent_fills_with_context = (
    fills_enriched.sort_values("fill_ts_utc", ascending=False)
    .merge(
        orders_enriched[["client_order_id", "quote_cycle_id", "action_type", "reason_code"]].drop_duplicates(),
        on=["client_order_id", "quote_cycle_id"],
        how="left",
        suffixes=("", "_order_action"),
    )
    .merge(markout_status_by_trade, on="trade_id", how="left")
)
recent_fills_with_context_display = recent_fills_with_context.assign(
    fill_ts_utc=format_utc(pd.to_datetime(recent_fills_with_context["fill_ts_utc"], utc=True, errors="coerce")),
    last_qty=format_number(recent_fills_with_context["last_qty_num"], digits=2),
    last_px=format_number(recent_fills_with_context["last_px_num"], digits=8),
    best_resolved_markout_bps=format_signed_number(recent_fills_with_context["best_resolved_markout_bps"], digits=2, suffix=" bps"),
)[[
    "strategy_id",
    "instrument_id",
    "client_order_id",
    "trade_id",
    "liquidity_side",
    "last_qty",
    "last_px",
    "action_type",
    "reason_code",
    "fv_source",
    "markout_rows",
    "resolved_rows",
    "expired_rows",
    "best_resolved_markout_bps",
    "fill_ts_utc",
]].head(25)

recent_fills_with_context_display


,strategy_id,instrument_id,client_order_id,trade_id,liquidity_side,last_qty,last_px,action_type,reason_code,fv_source,markout_rows,resolved_rows,expired_rows,best_resolved_markout_bps,fill_ts_utc
0,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,O-20260312-080909-SPOT-000-158,2270000001366793465,MAKER,1000.00,0.01211000,PLACE,None,fv_market_mid,3.0,3.0,0.0,+15.83 bps,2026-03-12 08:09:17 UTC
1,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,O-20260312-080910-PERP-000-4807,5d68cd58-ccf4-5c72-9369-a1bef89f4e53,MAKER,1000.00,0.01210000,PLACE,None,fv_market_mid,3.0,3.0,0.0,+18.39 bps,2026-03-12 08:09:17 UTC
2,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,O-20260312-080909-SPOT-000-157,2270000001366793464,MAKER,1000.00,0.01212000,PLACE,None,fv_market_mid,3.0,3.0,0.0,+7.56 bps,2026-03-12 08:09:17 UTC
3,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,O-20260312-080910-PERP-000-4806,889746cc-153b-5ab1-86fa-255788948a24,MAKER,1000.00,0.01210600,PLACE,None,fv_market_mid,3.0,3.0,0.0,+13.42 bps,2026-03-12 08:09:17 UTC
4,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,O-20260312-080910-SPOT-000-163,2270000001366793458,MAKER,1000.00,0.01213000,PLACE,None,fv_market_mid,3.0,3.0,0.0,-0.69 bps,2026-03-12 08:09:17 UTC
5,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,O-20260312-080910-PERP-000-4805,31d943bb-7727-544f-b034-974caadbd34d,MAKER,1000.00,0.01211200,PLACE,None,fv_market_mid,3.0,3.0,0.0,+8.46 bps,2026-03-12 08:09:17 UTC
6,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,O-20260312-080910-PERP-000-4804,9d834c64-ef28-52c7-8451-a80813a371db,MAKER,1000.00,0.01211800,PLACE,None,fv_market_mid,3.0,3.0,0.0,+3.51 bps,2026-03-12 08:09:17 UTC
7,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,O-20260312-080854-PERP-000-4654,f4040019-c8cb-53b9-8898-7449f7387cfd,MAKER,1000.00,0.01212900,PLACE,None,fv_market_mid,3.0,3.0,0.0,-7.49 bps,2026-03-12 08:09:02 UTC
8,plumeusdt_bybit_spot_makerv3-000,PLUMEUSDT-SPOT.BYBIT,O-20260312-080851-SPOT-000-132,2270000001366793258,MAKER,1000.00,0.01218000,PLACE,None,fv_market_mid,3.0,3.0,0.0,+41.05 bps,2026-03-12 08:08:54 UTC
9,plumeusdt_bybit_perp_makerv3-000,PLUMEUSDT-LINEAR.BYBIT,O-20260312-080846-PERP-000-4588,2a986ff8-70af-5539-ad6c-fb793de14edd,MAKER,1000.00,0.01217400,PLACE,None,fv_market_mid,3.0,3.0,0.0,+41.55 bps,2026-03-12 08:08:54 UTC


## Optional Postgres Swap

If the telemetry shipper is enabled later, swap the SQLite helpers for `pd.read_sql(...)` against the shipped telemetry tables. The reporting cells above can stay the same.
